# Visualisasi Hasil Implementasi dan Evaluasi Sistem RAG (Bab 4.3)
Notebook ini menghasilkan seluruh grafik untuk subbab 4.3 (Hasil Implementasi Sistem *Question Answering*
Berbasis RAG dan Evaluasinya — Tujuan 3), berdasarkan hasil pada `tujuan3/18-persiapan-evaluasi-rag.ipynb`,
`tujuan3/19-generate-prediction-RAG.ipynb`, `tujuan3/20-rag-evaluation.ipynb`, dan
`tujuan3/21-generate-scenario-evaluation.ipynb`. Data ringkasan (RAGAS per konfigurasi ablasi K1-K5,
delta antar-konfigurasi, dan RAGAS per skenario pengujian) disalin langsung dari output notebook-notebook
tersebut agar sel ini dapat dijalankan ulang tanpa bergantung pada file eksternal tambahan.

## Setup, pemuatan data, dan konfigurasi global

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.axisbelow": True, "figure.dpi": 150})

C_BASE, C_FT, C_K = "#b3b3b3", "#2c6fbb", "#e08a1e"
C_SIG, C_GOOD = "#c0392b", "#3a7d44"

nan = np.nan

# Ringkasan RAGAS per konfigurasi ablasi (K1-K5), sumber: tujuan3/20-rag-evaluation.ipynb
ragas_summary = pd.DataFrame([
    {"condition": "K1", "architecture": "gemma2",   "protocol": "end_to_end", "faithfulness": nan,      "answer_relevancy": 0.358900, "context_relevance": nan,      "n": 89, "n_answer_relevancy": 74},
    {"condition": "K1", "architecture": "llama3.2", "protocol": "end_to_end", "faithfulness": nan,      "answer_relevancy": 0.316934, "context_relevance": nan,      "n": 89, "n_answer_relevancy": 89},
    {"condition": "K2", "architecture": "gemma2",   "protocol": "controlled", "faithfulness": 0.779093, "answer_relevancy": 0.574682, "context_relevance": 0.963483, "n": 89, "n_answer_relevancy": 72},
    {"condition": "K2", "architecture": "gemma2",   "protocol": "end_to_end", "faithfulness": 0.810879, "answer_relevancy": 0.569866, "context_relevance": 0.949438, "n": 89, "n_answer_relevancy": 72},
    {"condition": "K2", "architecture": "llama3.2", "protocol": "controlled", "faithfulness": 0.795771, "answer_relevancy": 0.684764, "context_relevance": 0.957865, "n": 89, "n_answer_relevancy": 89},
    {"condition": "K2", "architecture": "llama3.2", "protocol": "end_to_end", "faithfulness": 0.759525, "answer_relevancy": 0.673227, "context_relevance": 0.963483, "n": 89, "n_answer_relevancy": 89},
    {"condition": "K3", "architecture": "gemma2",   "protocol": "end_to_end", "faithfulness": nan,      "answer_relevancy": 0.575942, "context_relevance": nan,      "n": 89, "n_answer_relevancy": 85},
    {"condition": "K3", "architecture": "llama3.2", "protocol": "end_to_end", "faithfulness": nan,      "answer_relevancy": 0.577945, "context_relevance": nan,      "n": 89, "n_answer_relevancy": 88},
    {"condition": "K4", "architecture": "gemma2",   "protocol": "controlled", "faithfulness": 0.822873, "answer_relevancy": 0.633022, "context_relevance": 0.957865, "n": 89, "n_answer_relevancy": 89},
    {"condition": "K4", "architecture": "gemma2",   "protocol": "end_to_end", "faithfulness": 0.803030, "answer_relevancy": 0.620667, "context_relevance": 0.954545, "n": 89, "n_answer_relevancy": 89},
    {"condition": "K4", "architecture": "llama3.2", "protocol": "controlled", "faithfulness": 0.831417, "answer_relevancy": 0.596216, "context_relevance": 0.960674, "n": 89, "n_answer_relevancy": 89},
    {"condition": "K4", "architecture": "llama3.2", "protocol": "end_to_end", "faithfulness": 0.826005, "answer_relevancy": 0.598264, "context_relevance": 0.966292, "n": 89, "n_answer_relevancy": 89},
    {"condition": "K5", "architecture": "deepseek", "protocol": "controlled", "faithfulness": 0.894637, "answer_relevancy": 0.464960, "context_relevance": 0.957865, "n": 89, "n_answer_relevancy": 89},
    {"condition": "K5", "architecture": "deepseek", "protocol": "end_to_end", "faithfulness": 0.920016, "answer_relevancy": 0.460855, "context_relevance": 0.952247, "n": 89, "n_answer_relevancy": 89},
])

# Delta berpasangan antar-konfigurasi (efek RAG dan efek fine-tuning), sumber: tujuan3/20-rag-evaluation.ipynb
ragas_deltas = pd.DataFrame([
    {"comparison": "K2-K1 (llama3.2, efek RAG di base)",             "answer_relevancy": 0.356294,  "faithfulness": nan,      "context_relevance": nan},
    {"comparison": "K4-K3 (llama3.2, efek RAG di fine-tuned)",       "answer_relevancy": 0.017125,  "faithfulness": nan,      "context_relevance": nan},
    {"comparison": "K4-K2 (llama3.2, efek fine-tuning, controlled)", "answer_relevancy": -0.088548, "faithfulness": 0.035647, "context_relevance": 0.002809},
    {"comparison": "K2-K1 (gemma2, efek RAG di base)",               "answer_relevancy": 0.200024,  "faithfulness": nan,      "context_relevance": nan},
    {"comparison": "K4-K3 (gemma2, efek RAG di fine-tuned)",         "answer_relevancy": 0.053141,  "faithfulness": nan,      "context_relevance": nan},
    {"comparison": "K4-K2 (gemma2, efek fine-tuning, controlled)",   "answer_relevancy": 0.074505,  "faithfulness": 0.043780, "context_relevance": -0.005618},
])

# Skor RAGAS per model x skenario pengujian (5 skenario, 20 soal), sumber: tujuan3/21-generate-scenario-evaluation.ipynb
scenario_ragas_by_model = pd.DataFrame([
    {"model_key": "deepseek-flash",      "scenario": "1_faktual_berbasis_narasi", "faithfulness_mean": 0.900000, "answer_relevancy_mean": 0.689726, "context_relevance_mean": 1.000000},
    {"model_key": "deepseek-flash",      "scenario": "2_interpretatif_ringan",    "faithfulness_mean": 1.000000, "answer_relevancy_mean": 0.668568, "context_relevance_mean": 1.000000},
    {"model_key": "deepseek-flash",      "scenario": "3_variasi_gaya_bahasa",     "faithfulness_mean": 0.714286, "answer_relevancy_mean": 0.477111, "context_relevance_mean": 1.000000},
    {"model_key": "deepseek-flash",      "scenario": "4_di_luar_konteks",         "faithfulness_mean": nan,      "answer_relevancy_mean": 0.000000, "context_relevance_mean": nan},
    {"model_key": "deepseek-flash",      "scenario": "5_kesalahan_ketik",         "faithfulness_mean": 1.000000, "answer_relevancy_mean": 0.479251, "context_relevance_mean": 1.000000},
    {"model_key": "gemma2-finetuned",    "scenario": "1_faktual_berbasis_narasi", "faithfulness_mean": 0.718750, "answer_relevancy_mean": 0.671003, "context_relevance_mean": 1.000000},
    {"model_key": "gemma2-finetuned",    "scenario": "2_interpretatif_ringan",    "faithfulness_mean": 0.854167, "answer_relevancy_mean": 0.469174, "context_relevance_mean": 1.000000},
    {"model_key": "gemma2-finetuned",    "scenario": "3_variasi_gaya_bahasa",     "faithfulness_mean": 0.968750, "answer_relevancy_mean": 0.606827, "context_relevance_mean": 1.000000},
    {"model_key": "gemma2-finetuned",    "scenario": "4_di_luar_konteks",         "faithfulness_mean": 1.000000, "answer_relevancy_mean": 0.000000, "context_relevance_mean": 0.250000},
    {"model_key": "gemma2-finetuned",    "scenario": "5_kesalahan_ketik",         "faithfulness_mean": 1.000000, "answer_relevancy_mean": 0.494628, "context_relevance_mean": 1.000000},
    {"model_key": "llama3.2-finetuned",  "scenario": "1_faktual_berbasis_narasi", "faithfulness_mean": 0.916667, "answer_relevancy_mean": 0.491477, "context_relevance_mean": 1.000000},
    {"model_key": "llama3.2-finetuned",  "scenario": "2_interpretatif_ringan",    "faithfulness_mean": 0.964286, "answer_relevancy_mean": 0.617623, "context_relevance_mean": 1.000000},
    {"model_key": "llama3.2-finetuned",  "scenario": "3_variasi_gaya_bahasa",     "faithfulness_mean": 0.708333, "answer_relevancy_mean": 0.636908, "context_relevance_mean": 1.000000},
    {"model_key": "llama3.2-finetuned",  "scenario": "4_di_luar_konteks",         "faithfulness_mean": 1.000000, "answer_relevancy_mean": 0.000000, "context_relevance_mean": 0.000000},
    {"model_key": "llama3.2-finetuned",  "scenario": "5_kesalahan_ketik",         "faithfulness_mean": 0.916667, "answer_relevancy_mean": 0.398821, "context_relevance_mean": 1.000000},
])

ragas_summary.head()

## Fungsi pembuat grafik (satu sel per grafik)

### Fig 1 — Diagram arsitektur alur pemrosesan sistem QA berbasis RAG
Diagram digambar ulang dari kode produksi aktual (`app/retrieval_service.py`, `app/reranker_service.py`,
`app/query_expansion_service.py`, `app/main.py`), bukan dari rancangan awal pada Bab 3, sehingga
merepresentasikan pipeline hibrida (dense+BM25 -> RRF -> rerank -> sitasi/generasi) yang sesungguhnya
diimplementasikan.

In [ ]:
def fig1_arsitektur_pipeline():
    EDGE = "#333333"
    C_UI, C_API, C_RET, C_GEN, C_DATA = "#e8eef7", "#fdf1e0", "#e5f2e8", "#f3e6f0", "#eeeeee"
    FIG_W, FIG_H = 15.5, 9.5
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.set_xlim(0, FIG_W); ax.set_ylim(0, FIG_H); ax.axis("off")

    def box(x, y, w, h, text, color, fontsize=9):
        b = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                            linewidth=1.1, edgecolor=EDGE, facecolor=color, zorder=2)
        ax.add_patch(b)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True, zorder=3)
        return (x, y, w, h)

    def arrow(b1, b2, side1="right", side2="left", text=None, style="-|>", color=EDGE, curve=0.0, lw=1.1):
        x1, y1, w1, h1 = b1; x2, y2, w2, h2 = b2
        pts = {"right": (x1 + w1, y1 + h1 / 2), "left": (x1, y1 + h1 / 2),
               "top": (x1 + w1 / 2, y1 + h1), "bottom": (x1 + w1 / 2, y1)}
        pts2 = {"right": (x2 + w2, y2 + h2 / 2), "left": (x2, y2 + h2 / 2),
                "top": (x2 + w2 / 2, y2 + h2), "bottom": (x2 + w2 / 2, y2)}
        p1, p2 = pts[side1], pts2[side2]
        a = FancyArrowPatch(p1, p2, arrowstyle=style, mutation_scale=12,
                             connectionstyle=f"arc3,rad={curve}", color=color, linewidth=lw, zorder=1)
        ax.add_patch(a)
        if text:
            mx, my = (p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2 + 0.14
            ax.text(mx, my, text, ha="center", va="bottom", fontsize=7.6, color="#222", zorder=4)

    b_user = box(0.4, 8.0, 2.1, 0.85, "Pengguna\n(Web UI)", C_UI)
    b_api = box(3.1, 8.0, 2.6, 0.85, "FastAPI Backend\n/api/ask(/stream)", C_API, fontsize=8.6)
    arrow(b_user, b_api, "right", "left", text="pertanyaan")
    b_queue = box(11.8, 8.0, 3.2, 0.85, "Antrean Permintaan\n(semaphore, QUEUE_CONCURRENCY)\n\u2014 batasi request paralel", C_DATA, fontsize=7.6)

    b_disp = box(0.4, 6.5, 2.1, 0.85, "Tampilan:\njawaban + kartu\nsitasi per sumber", C_UI, fontsize=8.4)
    b_qe = box(3.1, 6.5, 2.6, 0.85, "Query Expansion\n(SLM aktif)", C_RET, fontsize=8.8)
    b_answer = box(9.1, 6.5, 3.0, 0.85, "Jawaban + Daftar Sumber\n(JSON / SSE stage events)", C_API, fontsize=8.2)
    arrow(b_api, b_disp, "left", "right", curve=0.35)
    arrow(b_disp, b_user, "top", "bottom", curve=-0.35)
    arrow(b_api, b_qe, "bottom", "top", curve=0.0)
    arrow(b_answer, b_api, "left", "right", curve=0.28)

    b_dense = box(0.4, 4.9, 2.7, 0.95, "Dense Retrieval\nnomic-embed-text-v1.5\n+ FAISS IndexFlatIP (top-20)", C_RET, fontsize=8.0)
    b_sparse = box(3.4, 4.9, 2.3, 0.95, "Sparse Retrieval\nBM25 (top-20)", C_RET, fontsize=8.6)
    arrow(b_qe, b_dense, "bottom", "top", curve=-0.25)
    arrow(b_qe, b_sparse, "bottom", "top", curve=0.1)

    b_kb = box(0.4, 3.3, 2.7, 0.85, "Basis Pengetahuan\nfaiss_index.bin +\nchunk_metadata.jsonl", C_DATA, fontsize=7.8)
    b_rrf = box(3.4, 3.3, 2.3, 0.85, "Reciprocal Rank\nFusion (k=60)", C_RET, fontsize=8.6)
    arrow(b_kb, b_dense, "top", "bottom")
    arrow(b_dense, b_rrf, "bottom", "top", curve=0.15)
    arrow(b_sparse, b_rrf, "bottom", "top", curve=-0.1)

    b_rerank = box(3.1, 1.5, 2.9, 0.85, "Cross-Encoder Rerank\nmadebyaris/rerank-indonesia\n(ambang skor, top-5)", C_RET, fontsize=7.6)
    b_cite = box(6.6, 1.5, 2.5, 0.85, "Citation Service\n(sitasi ringkas per\nsumber, JSON)", C_GEN, fontsize=7.8)
    b_prompt = box(9.6, 1.5, 2.5, 0.85, "Prompt Constructor\n(Konteks + Pertanyaan)", C_GEN, fontsize=8.2)
    b_gen = box(12.6, 1.5, 2.4, 0.85, "Generation\nService", C_GEN, fontsize=9)
    arrow(b_rrf, b_rerank, "bottom", "top", text="20 kandidat")
    arrow(b_rerank, b_cite, "right", "left", text="5 sumber")
    arrow(b_rerank, b_prompt, "top", "top", curve=-0.45)
    arrow(b_prompt, b_gen, "right", "left")

    b_ollama = box(12.6, 3.3, 2.4, 0.85, "Server Inferensi\nOllama\n(GGUF Q4_K_M, lokal)", C_DATA, fontsize=7.8)
    arrow(b_gen, b_ollama, "top", "bottom", style="<|-|>", text="HTTP")
    arrow(b_cite, b_answer, "top", "bottom", curve=0.25)
    arrow(b_gen, b_answer, "left", "right", curve=0.3)

    ax.text(FIG_W / 2, 9.15, "Alur Pemrosesan Satu Pertanyaan pada Sistem QA Berbasis RAG",
            ha="center", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("fig1_arsitektur_pipeline.png", dpi=200, bbox_inches="tight")
    plt.show()

fig1_arsitektur_pipeline()

### Fig 2 — Skor RAGAS pada lima konfigurasi ablasi (K1-K5, protokol end-to-end)
K1 = model dasar tanpa RAG, K2 = model dasar + RAG, K3 = model fine-tuned tanpa RAG,
K4 = model fine-tuned + RAG, K5 = LLM komersial (deepseek-flash) + RAG sebagai *upper bound*.

In [ ]:
def fig2_ragas_ablasi():
    d = ragas_summary[ragas_summary["protocol"] == "end_to_end"].copy()
    order = ["K1\ngemma2", "K1\nllama3.2", "K2\ngemma2", "K2\nllama3.2",
             "K3\ngemma2", "K3\nllama3.2", "K4\ngemma2", "K4\nllama3.2", "K5\ndeepseek"]
    d["label"] = d["condition"] + "\n" + d["architecture"]
    d["order"] = d["label"].apply(order.index)
    d = d.sort_values("order")
    d["xlabel"] = d["condition"] + " " + d["architecture"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=True)
    metrics = [("faithfulness", "Faithfulness"), ("answer_relevancy", "Answer Relevancy"),
               ("context_relevance", "Context Relevance")]
    colors = [C_SIG, C_SIG, C_K, C_K, C_BASE, C_BASE, C_FT, C_FT, C_GOOD]
    for ax, (col, title) in zip(axes, metrics):
        vals = d[col].values
        ax.bar(range(len(d)), np.nan_to_num(vals, nan=0), color=colors, edgecolor="black", linewidth=.5)
        for i, v in enumerate(vals):
            if not np.isnan(v):
                ax.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=7.5)
            else:
                ax.text(i, 0.02, "n/a", ha="center", fontsize=7, color="gray", rotation=90)
        ax.set_xticks(range(len(d)))
        ax.set_xticklabels(d["xlabel"], fontsize=7.5, rotation=40, ha="right")
        ax.set_title(title); ax.set_ylim(0, 1.05)
    axes[0].set_ylabel("Skor RAGAS")
    fig.suptitle("Skor RAGAS pada Lima Konfigurasi Ablasi (K1-K5, protokol end-to-end)", y=1.02)
    plt.tight_layout()
    plt.savefig("fig2_ragas_ablasi.png", bbox_inches="tight")
    plt.show()

fig2_ragas_ablasi()

### Fig 3 — Kontribusi RAG dan fine-tuning terhadap Answer Relevancy
Selisih skor RAGAS antar pasangan konfigurasi yang berdekatan untuk mengisolasi efek RAG (K2-K1, K4-K3)
dari efek fine-tuning (K4-K2, protokol *controlled* agar konteks retrieval yang diberikan identik).

In [ ]:
def fig3_ragas_delta():
    d = ragas_deltas.copy()
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    y = np.arange(len(d))
    ax.barh(y, d["answer_relevancy"], color=C_FT, edgecolor="black", linewidth=.5, label="\u0394 Answer Relevancy")
    xmin, xmax = d["answer_relevancy"].min(), d["answer_relevancy"].max()
    pad = (xmax - xmin) * 0.18
    ax.set_xlim(xmin - pad, xmax + pad)
    for yi, v in zip(y, d["answer_relevancy"]):
        ax.text(v + (0.008 if v >= 0 else -0.008), yi, f"{v:+.3f}", va="center",
                ha="left" if v >= 0 else "right", fontsize=8)
    ax.axvline(0, color="black", lw=.8)
    ax.set_yticks(y); ax.set_yticklabels(d["comparison"], fontsize=8.5)
    ax.set_xlabel("Selisih Skor (kondisi kedua \u2212 kondisi pertama)")
    ax.set_title("Kontribusi RAG dan Fine-Tuning terhadap Answer Relevancy")
    plt.tight_layout()
    plt.savefig("fig3_ragas_delta.png", bbox_inches="tight")
    plt.show()

fig3_ragas_delta()

### Fig 4 — Skor RAGAS pada lima skenario pengujian per model
Pengujian ketahanan dengan 20 pertanyaan tetap (4 per skenario) terhadap tiga model produksi:
Llama-3.2 (fine-tuned), Gemma-2 (fine-tuned), dan DeepSeek-Flash (komersial, pembanding).

In [ ]:
def fig4_skenario_heatmap():
    d = scenario_ragas_by_model.copy()
    scenario_labels = {
        "1_faktual_berbasis_narasi": "1. Faktual", "2_interpretatif_ringan": "2. Interpretatif",
        "3_variasi_gaya_bahasa": "3. Variasi gaya", "4_di_luar_konteks": "4. Luar konteks",
        "5_kesalahan_ketik": "5. Kesalahan ketik",
    }
    model_labels = {"llama3.2-finetuned": "Llama-3.2 (fine-tuned)", "gemma2-finetuned": "Gemma-2 (fine-tuned)",
                    "deepseek-flash": "DeepSeek-Flash (komersial)"}
    model_order = ["llama3.2-finetuned", "gemma2-finetuned", "deepseek-flash"]
    scen_order = list(scenario_labels.keys())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, (col, title) in zip(axes, [("faithfulness_mean", "Faithfulness"), ("answer_relevancy_mean", "Answer Relevancy")]):
        mat = np.zeros((len(model_order), len(scen_order)))
        for i, m in enumerate(model_order):
            for j, s in enumerate(scen_order):
                row = d[(d["model_key"] == m) & (d["scenario"] == s)]
                mat[i, j] = row[col].values[0] if len(row) else np.nan
        ax.imshow(mat, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(scen_order))); ax.set_xticklabels([scenario_labels[s] for s in scen_order], rotation=20, ha="right", fontsize=8)
        ax.set_yticks(range(len(model_order))); ax.set_yticklabels([model_labels[m] for m in model_order], fontsize=8.5)
        for i in range(len(model_order)):
            for j in range(len(scen_order)):
                v = mat[i, j]
                label = "n/a" if np.isnan(v) else f"{v:.2f}"
                ax.text(j, i, label, ha="center", va="center", fontsize=8.5,
                        color="white" if (not np.isnan(v) and v > 0.55) else "black")
        ax.set_title(title)
    fig.suptitle("Skor RAGAS pada Lima Skenario Pengujian per Model", y=1.04)
    plt.tight_layout()
    plt.savefig("fig4_skenario_heatmap.png", bbox_inches="tight")
    plt.show()

fig4_skenario_heatmap()